# 01 — Data Cleaning and Preparation

This notebook is the first stage of the dynamic-pricing project. It loads and audits the Kaggle ridesharing dataset, cleans the schema, creates useful features, and produces reproducible training and testing files for the later notebooks.

**Outputs written to `artifacts/`:**

- `cleaned_dynamic_pricing.csv`
- `train_features.csv` and `test_features.csv`
- `train_target.csv` and `test_target.csv`
- `data_manifest.json`

Run this notebook before notebooks 02–04.

## 1. Imports and paths

In [1]:
from pathlib import Path
import json
import random
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'dynamic_pricing.csv').exists() and (PROJECT_ROOT.parent / 'dynamic_pricing.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
print(f'Project root: {PROJECT_ROOT.resolve()}')

Project root: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Artificial Intellegence Driven Decision Making\CA2 Project


## 2. Load and inspect the raw data

The inspection displays shape, columns, data types, missing values, duplicate count, and descriptive statistics before any changes are made.

In [2]:
csv_path = PROJECT_ROOT / 'dynamic_pricing.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'Expected dataset at {csv_path.resolve()}')

raw_df = pd.read_csv(csv_path)
print(f'Dataset shape: {raw_df.shape}')
print(f'Duplicate rows: {raw_df.duplicated().sum()}')
display(raw_df.head())

Dataset shape: (1000, 10)
Duplicate rows: 0


,Number_of_Riders,Number_of_Drivers,Location_Category,Customer_Loyalty_Status,Number_of_Past_Rides,Average_Ratings,Time_of_Booking,Vehicle_Type,Expected_Ride_Duration,Historical_Cost_of_Ride
0,90,45,Urban,Silver,13,4.47,Night,Premium,90,284.257273
1,58,39,Suburban,Silver,72,4.06,Evening,Economy,43,173.874753
2,42,31,Rural,Silver,0,3.99,Afternoon,Premium,76,329.795469
3,89,28,Rural,Regular,67,4.31,Afternoon,Premium,134,470.201232
4,78,22,Rural,Regular,74,3.77,Afternoon,Economy,149,579.681422


In [3]:
print('Columns:')
print(raw_df.columns.tolist())

print('\nData types:')
display(raw_df.dtypes.to_frame('dtype'))

print('Missing values:')
display(raw_df.isna().sum().to_frame('missing_values'))

print('Summary statistics:')
display(raw_df.describe(include='all').T)

Columns:
['Number_of_Riders', 'Number_of_Drivers', 'Location_Category', 'Customer_Loyalty_Status', 'Number_of_Past_Rides', 'Average_Ratings', 'Time_of_Booking', 'Vehicle_Type', 'Expected_Ride_Duration', 'Historical_Cost_of_Ride']

Data types:


,dtype
Number_of_Riders,int64
Number_of_Drivers,int64
Location_Category,str
Customer_Loyalty_Status,str
Number_of_Past_Rides,int64
Average_Ratings,float64
Time_of_Booking,str
Vehicle_Type,str
Expected_Ride_Duration,int64
Historical_Cost_of_Ride,float64


Missing values:


,missing_values
Number_of_Riders,0
Number_of_Drivers,0
Location_Category,0
Customer_Loyalty_Status,0
Number_of_Past_Rides,0
Average_Ratings,0
Time_of_Booking,0
Vehicle_Type,0
Expected_Ride_Duration,0
Historical_Cost_of_Ride,0


Summary statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Number_of_Riders,1000.0,NaN,NaN,NaN,60.372,23.701506,20.0,40.0,60.0,81.0,100.0
Number_of_Drivers,1000.0,NaN,NaN,NaN,27.076,19.068346,5.0,11.0,22.0,38.0,89.0
Location_Category,1000,3,Urban,346,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_Loyalty_Status,1000,3,Silver,367,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Number_of_Past_Rides,1000.0,NaN,NaN,NaN,50.031,29.313774,0.0,25.0,51.0,75.0,100.0
Average_Ratings,1000.0,NaN,NaN,NaN,4.25722,0.435781,3.5,3.87,4.27,4.6325,5.0
Time_of_Booking,1000,4,Night,276,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vehicle_Type,1000,2,Premium,522,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Expected_Ride_Duration,1000.0,NaN,NaN,NaN,99.588,49.16545,10.0,59.75,102.0,143.0,180.0
Historical_Cost_of_Ride,1000.0,NaN,NaN,NaN,372.502623,187.158756,25.993449,221.365202,362.019426,510.497504,836.116419


## 3. Clean column names and values

Column names are converted to `snake_case`, exact duplicate rows are removed, blank categorical strings are treated as missing, and invalid infinite numeric values are converted to missing values. Leakage-safe median/mode imputation is deliberately fitted in notebook 02 using training data only.

In [4]:
df = raw_df.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r'[^a-z0-9]+', '_', regex=True)
              .str.strip('_'))

duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

categorical_columns = df.select_dtypes(include=['object', 'string']).columns
for column in categorical_columns:
    df[column] = df[column].astype('string').str.strip().replace('', pd.NA)

numeric_columns = df.select_dtypes(include=np.number).columns
df[numeric_columns] = df[numeric_columns].replace([np.inf, -np.inf], np.nan)

print(f'Removed {duplicate_count} duplicate row(s).')
print(f'Cleaned shape: {df.shape}')
display(df.isna().sum().to_frame('missing_after_cleaning'))

Removed 0 duplicate row(s).
Cleaned shape: (1000, 10)


,missing_after_cleaning
number_of_riders,0
number_of_drivers,0
location_category,0
customer_loyalty_status,0
number_of_past_rides,0
average_ratings,0
time_of_booking,0
vehicle_type,0
expected_ride_duration,0
historical_cost_of_ride,0


## 4. Feature engineering

The most important engineered feature measures demand pressure:

$$\text{demand supply ratio}=\frac{\text{number of riders}}{\max(\text{number of drivers},1)}$$

Riders per minute captures demand relative to trip length. The experienced-customer flag provides a simple customer-history segment.

In [5]:
df['demand_supply_ratio'] = (
    df['number_of_riders'] / df['number_of_drivers'].clip(lower=1)
)
df['riders_per_minute'] = (
    df['number_of_riders'] / df['expected_ride_duration'].clip(lower=1)
)
df['experienced_customer'] = (
    df['number_of_past_rides'] >= df['number_of_past_rides'].median()
).astype(int)

display(df.head())
display(df[['demand_supply_ratio', 'riders_per_minute']].describe().T)

,number_of_riders,number_of_drivers,location_category,customer_loyalty_status,number_of_past_rides,average_ratings,time_of_booking,vehicle_type,expected_ride_duration,historical_cost_of_ride,demand_supply_ratio,riders_per_minute,experienced_customer
0,90,45,Urban,Silver,13,4.47,Night,Premium,90,284.257273,2.000000,1.000000,0
1,58,39,Suburban,Silver,72,4.06,Evening,Economy,43,173.874753,1.487179,1.348837,1
2,42,31,Rural,Silver,0,3.99,Afternoon,Premium,76,329.795469,1.354839,0.552632,0
3,89,28,Rural,Regular,67,4.31,Afternoon,Premium,134,470.201232,3.178571,0.664179,1
4,78,22,Rural,Regular,74,3.77,Afternoon,Economy,149,579.681422,3.545455,0.523490,1


,count,mean,std,min,25%,50%,75%,max
demand_supply_ratio,1000.0,3.235461,2.533519,1.112360,1.658793,2.357143,3.800000,17.6
riders_per_minute,1000.0,0.994620,1.168348,0.116667,0.375414,0.598165,1.052529,9.1


## 5. Define features and split the data

A fixed random seed produces the same 80/20 split every time. The target is historical ride cost, used as a proxy for expected base ride revenue.

In [6]:
from sklearn.model_selection import train_test_split

TARGET = 'historical_cost_of_ride'
FEATURE_COLUMNS = [
    'number_of_riders', 'number_of_drivers', 'demand_supply_ratio',
    'riders_per_minute', 'expected_ride_duration', 'number_of_past_rides',
    'average_ratings', 'experienced_customer', 'location_category',
    'customer_loyalty_status', 'time_of_booking', 'vehicle_type'
]

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

# Reset indices so every saved feature row aligns with its saved target row.
X_train, X_test = X_train.reset_index(drop=True), X_test.reset_index(drop=True)
y_train, y_test = y_train.reset_index(drop=True), y_test.reset_index(drop=True)

print(f'Training set: {X_train.shape}')
print(f'Testing set:  {X_test.shape}')
print(f'Target train/test: {y_train.shape} / {y_test.shape}')

Training set: (800, 12)
Testing set:  (200, 12)
Target train/test: (800,) / (200,)


## 6. Save prepared data for later notebooks

In [7]:
df.to_csv(ARTIFACT_DIR / 'cleaned_dynamic_pricing.csv', index=False)
X_train.to_csv(ARTIFACT_DIR / 'train_features.csv', index=False)
X_test.to_csv(ARTIFACT_DIR / 'test_features.csv', index=False)
y_train.to_frame(TARGET).to_csv(ARTIFACT_DIR / 'train_target.csv', index=False)
y_test.to_frame(TARGET).to_csv(ARTIFACT_DIR / 'test_target.csv', index=False)

manifest = {
    'random_seed': SEED,
    'test_size': 0.20,
    'target': TARGET,
    'feature_columns': FEATURE_COLUMNS,
    'training_rows': len(X_train),
    'testing_rows': len(X_test),
}
with open(ARTIFACT_DIR / 'data_manifest.json', 'w', encoding='utf-8') as file:
    json.dump(manifest, file, indent=2)

print('Saved preparation artifacts:')
for path in sorted(ARTIFACT_DIR.glob('*')):
    print(f' - {path.name}')

Saved preparation artifacts:
 - .gitkeep
 - cleaned_dynamic_pricing.csv
 - data_manifest.json
 - model_metrics.csv
 - policy_comparison.csv
 - policy_level_results.csv
 - q_learning_artifacts.joblib
 - random_forest_revenue_model.joblib
 - test_environment_costs.csv
 - test_features.csv
 - test_target.csv
 - train_environment_costs.csv
 - train_features.csv
 - train_target.csv
 - training_rewards.csv


## Stage complete

Data cleaning, feature engineering, and splitting are complete. Continue with **02_random_forest_training_evaluation.ipynb**.